# Validación de métricas: MIRD (medido) vs ISM (simulado)

**Objetivo.** Comprobar si el simulador ISM calibrado reproduce, en **métricas de beamforming** (PESQ / STOI / SDR / SIR / SAR), lo que se obtiene con las RIRs **medidas** del dataset MIRD (Hadad et al., IWAENC 2014), bajo **geometría y condiciones acústicas idénticas**.

**Estrategia.** Se llama a los dos benchmarks con la *misma* configuración:
- `run_mird_grid_search` → RIRs medidas del dataset MIRD.
- `run_grid_search` con `geometry_mode='mird_linear'` → RIRs **simuladas** por ISM, replicando exactamente el array lineal MIRD (`4-4-4-8-4-4-4`), la sala (6×6×2.4 m) y la colocación fuente/interferencia por ángulo/distancia.

**Condiciones fijas del dataset MIRD** (Tabla 1 del paper):
- Sala 6×6×2.4 m, array centrado; RT60 ∈ {160, 360, 610} ms.
- Array lineal de 8 mics, spacings {3-3-3-8-3-3-3, 4-4-4-8-4-4-4, 8-8-8-8-8-8-8} cm.
- Altavoces en semicírculos a 1 m y 2 m, ángulos −90°..90° cada 15°.

**Prueba singular (este notebook):** RT60=610 ms, spacing `4-4-4-8-4-4-4`, target 0°/1 m, interferencia 45°/1 m, iSIR=0 dB, sin mismatch de HW, sin WPE, sin error de apuntamiento. Una vez validado este punto, se escala a más RT / ángulos / spacings.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Localizar la raíz del proyecto y agregar src/ al path ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'src')):
    # Fallback por si el cwd del kernel no es tests/ism_validation
    PROJECT_ROOT = '/home/matias/Documents/Tesis/Vision-Aided-Beamformer'
SRC = os.path.join(PROJECT_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(PROJECT_ROOT)  # para que las rutas relativas (tools/data/...) resuelvan
print('PROJECT_ROOT =', PROJECT_ROOT)

from evaluation.full_benchmark_test_dtln import run_grid_search
from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from propagation.mird_loader import MirdDatasetProvider, generate_mird_linear_array
from evaluation.bf_wrappers import DS, NM_MVDR, ORACLE_MB_MVDR_SOUDEN
print('Imports OK')

## 1. Configuración compartida

Un único punto de operación. `param_grid` es **idéntico** para ambos benchmarks (mismos ejes: `rt60`, `target_angle`, `target_dist`, `interf_configs`, ...). Lo único que cambia es cómo se generan las RIRs (medidas vs simuladas), controlado por `geometry_mode` en la config ISM.

In [ ]:
# ---------------- Punto de operación (editar para escalar) ----------------
FS            = 16000
DURATION      = 10           # s (igual en ambos; sube a 15 si querés más resolución de métricas)
RT60          = 0.610        # s
SPACING       = '4-4-4-8-4-4-4'   # spacing que coincide con generate_mird_linear_array()
TARGET_ANGLE  = 0            # grados (0 = broadside)
TARGET_DIST   = 1.0          # m
INTERF_CONFIG = [(45, 1.0)]  # lista de (ángulo, distancia) por interferencia
ISIR_DB       = 0            # dB
EVAL_REFS     = ['anechoic', 'early', 'reverberant']

SOURCE  = os.path.join(PROJECT_ROOT, 'tools/data/signals/p002_emo_adoration_sentences.wav')
INTERF  = [os.path.join(PROJECT_ROOT, 'tools/data/signals/techno_gated commune.wav')]
DTLN_M1 = os.path.join(PROJECT_ROOT, 'src/dnn_denoise/models/model_quant_1.tflite')
MIRD_ROOT = os.path.join(PROJECT_ROOT, 'tools/data/rirs/mird')

OUT_MIRD = os.path.join(PROJECT_ROOT, 'tests/ism_validation/out_mird')
OUT_ISM  = os.path.join(PROJECT_ROOT, 'tests/ism_validation/out_ism')

# ---------------- Grilla común (mismos ejes para ambos) ----------------
param_grid = {
    'rt60':            [RT60],
    'target_angle':    [TARGET_ANGLE],
    'target_dist':     [TARGET_DIST],
    'interf_configs':  [INTERF_CONFIG],
    'isir_db':         [ISIR_DB],
    'mismatch_gain':   [0],
    'mismatch_phase':  [0],
    'use_wpe':         [False],
    'error_angle_deg': [0.0],
    'error_distance_m':[0.0],
}

# ---------------- Config base (campos comunes) ----------------
def make_base_config():
    return {
        'fs': FS, 'duration': DURATION, 't_early': 0.050,
        'array_center': [3.0, 3.0, 1.2],
        'snr_db': 60.0,
        'source_path': SOURCE, 'interf_paths': INTERF,
        'wpe_taps': 7, 'wpe_delay': 3, 'wpe_alpha': 0.9999,
        'wpe_stft_size': 512, 'wpe_stft_shift': 128,
        'stft_window': 512, 'stft_overlap': 384,
        'eval_references': EVAL_REFS,
        'dtln_model_path': DTLN_M1,
    }

# Config MIRD (RIRs medidas)
cfg_mird = make_base_config()
cfg_mird['mird_spacing'] = SPACING

# Config ISM (RIRs simuladas, réplica geométrica de MIRD)
cfg_ism = make_base_config()
cfg_ism['geometry_mode'] = 'mird_linear'
cfg_ism['room_dims']     = [6.0, 6.0, 2.4]
cfg_ism['ray_tracing']   = False   # el modelo ISM calibrado (pure-ISM order 50) valida vs MIRD

# ---------------- Procesadores (idénticos para ambos) ----------------
def make_processors():
    return {
        'DS': DS(),
        'NM-MVDR': NM_MVDR(min_loading=1e-6, alpha=0.99),
        'Oracle-MVDR': ORACLE_MB_MVDR_SOUDEN(min_loading=1e-6, alpha=0.99, sharpen_exp=1.0),
    }

print('Config lista. eval_references =', EVAL_REFS)

## 2. Benchmark MIRD (RIRs medidas)

Sin DTLN post/mono (`interpreter_*=None`) y sin catálogo (`save_catalog=False`) para que sea rápido. NM-MVDR sigue usando la máscara neuronal internamente vía `dtln_model_path`.

In [ ]:
provider = MirdDatasetProvider(root_dir=MIRD_ROOT)

df_mird = run_mird_grid_search(
    grid_params=param_grid,
    dataset_provider=provider,
    processors=make_processors(),
    scene_base_config=cfg_mird,
    output_dir=OUT_MIRD,
    interpreter_1=None, interpreter_2=None,
    save_catalog=False, apply_dtln_post=False,
)
print('\nMIRD listo:', df_mird.shape)
df_mird[['processor', 'proc_SIR_early', 'proc_SDR_early', 'proc_STOI_early', 'proc_PESQ_early']]

## 3. Benchmark ISM (RIRs simuladas, geometría MIRD)

`room_profiles=None`: en modo `mird_linear` la sala sale de `cfg_ism['room_dims']`, no de un perfil.

In [ ]:
df_ism = run_grid_search(
    grid_params=param_grid,
    room_profiles=None,
    processors=make_processors(),
    scene_base_config=cfg_ism,
    output_dir=OUT_ISM,
    interpreter_1=None, interpreter_2=None,
    save_catalog=False, apply_dtln_post=False,
)
print('\nISM listo:', df_ism.shape)
df_ism[['processor', 'proc_SIR_early', 'proc_SDR_early', 'proc_STOI_early', 'proc_PESQ_early']]

## 4. Chequeo de geometría idéntica

Confirmamos que ambos benchmarks usaron exactamente el mismo array y las mismas posiciones de fuente/interferencia (sólo cambian las RIRs).

In [ ]:
def geom_summary(cfg, name):
    mc = np.asarray(cfg['mic_coords'])
    print(f'[{name}] mic_coords shape={mc.shape}  y-span={mc[:,1].min():.3f}..{mc[:,1].max():.3f} m')
    print(f'[{name}] source_pos = {np.asarray(cfg["source_pos"]).ravel()}')
    print(f'[{name}] interf_pos = {np.asarray(cfg["interferences_pos"]).tolist()}')

geom_summary(cfg_mird, 'MIRD')
geom_summary(cfg_ism,  'ISM ')

assert np.allclose(cfg_mird['mic_coords'], cfg_ism['mic_coords']), 'Los arrays difieren!'
assert np.allclose(np.asarray(cfg_mird['source_pos']).ravel(), np.asarray(cfg_ism['source_pos']).ravel())
print('\nOK: geometría idéntica entre MIRD e ISM.')

## 5. Tabla comparativa de métricas

Para cada procesador comparamos, por referencia (`anechoic`/`early`/`reverberant`):
- **base_**: métrica del micrófono de referencia (mezcla degradada, pre-beamforming).
- **proc_**: métrica de salida del beamformer.
- **Delta_tot_**: mejora (proc − base).

La diferencia `ISM − MIRD` cuantifica cuánto se aparta el simulado del medido.

In [ ]:
METRICS = ['PESQ', 'STOI', 'SDR', 'SIR', 'SAR']
KINDS   = ['base', 'proc', 'Delta_tot']

m = df_mird.set_index('processor')
s = df_ism.set_index('processor')
procs = list(m.index)

rows = []
for proc in procs:
    for ref in EVAL_REFS:
        for met in METRICS:
            for kind in KINDS:
                col = f'{kind}_{met}_{ref}'
                if col in m.columns and col in s.columns:
                    vm, vi = m.loc[proc, col], s.loc[proc, col]
                    rows.append({
                        'processor': proc, 'ref': ref, 'metric': met, 'kind': kind,
                        'MIRD': vm, 'ISM': vi, 'ISM-MIRD': vi - vm,
                    })
cmp = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda x: f'{x:8.3f}')
print('Comparación (ref=early):')
cmp[cmp.ref == 'early'].pivot_table(index=['processor', 'metric'], columns='kind',
                                    values=['MIRD', 'ISM', 'ISM-MIRD'])

In [ ]:
# Resumen: error absoluto medio ISM vs MIRD por métrica (salida del beamformer, ref=early)
sub = cmp[(cmp.kind == 'proc') & (cmp.ref == 'early')]
print('MAE(|ISM - MIRD|) por métrica (proc, early):')
print(sub.groupby('metric')['ISM-MIRD'].apply(lambda x: np.mean(np.abs(x))).round(3).to_string())

## 6. Gráficos comparativos

In [ ]:
def grouped_bars(kind, ref='early', title=None):
    """Barras MIRD vs ISM: una subfigura por métrica, barras por procesador."""
    data = cmp[(cmp['kind'] == kind) & (cmp['ref'] == ref)]
    fig, axes = plt.subplots(1, len(METRICS), figsize=(3.1 * len(METRICS), 3.8), squeeze=False)
    x = np.arange(len(procs)); w = 0.38
    for ax, met in zip(axes[0], METRICS):
        d = data[data['metric'] == met].set_index('processor')
        vm = [d.loc[p, 'MIRD'] if p in d.index else np.nan for p in procs]
        vi = [d.loc[p, 'ISM']  if p in d.index else np.nan for p in procs]
        ax.bar(x - w/2, vm, w, label='MIRD (medido)', color='#2c7fb8')
        ax.bar(x + w/2, vi, w, label='ISM (simulado)', color='#e6550d')
        ax.set_title(met)
        ax.set_xticks(x); ax.set_xticklabels(procs, rotation=30, ha='right', fontsize=8)
        ax.grid(axis='y', ls=':', alpha=0.6)
    axes[0][0].legend(fontsize=8, loc='best')
    fig.suptitle(title or f'{kind}  (ref={ref})', y=1.02, fontsize=12)
    fig.tight_layout()
    return fig

grouped_bars('proc',      title='Salida del beamformer: MIRD vs ISM')
grouped_bars('Delta_tot', title='Mejora (Delta_tot = proc - base): MIRD vs ISM')
plt.show()

In [ ]:
# Diferencia ISM - MIRD (métrica de salida) por procesador y métrica, todas las refs
fig, axes = plt.subplots(1, len(EVAL_REFS), figsize=(4.2 * len(EVAL_REFS), 3.8), squeeze=False)
for ax, ref in zip(axes[0], EVAL_REFS):
    d = cmp[(cmp['kind'] == 'proc') & (cmp['ref'] == ref)]
    piv = d.pivot(index='metric', columns='processor', values='ISM-MIRD').reindex(METRICS)
    piv.plot(kind='bar', ax=ax, legend=(ref == EVAL_REFS[0]))
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'ISM - MIRD (ref={ref})'); ax.set_ylabel('Δ métrica')
    ax.grid(axis='y', ls=':', alpha=0.6)
fig.suptitle('Discrepancia simulado - medido (0 = réplica perfecta)', y=1.02)
fig.tight_layout()
plt.show()

## 6b. Comparación en términos de la MEJORA del procesador (`Delta_tot`)

El bloque anterior compara el **valor absoluto** de salida del beamformer. Acá repetimos la comparación `|ISM − MIRD|` pero sobre la **mejora que aporta el procesador**, `Delta_tot = proc − base` (salida menos micrófono de referencia).

Esto es más justo para validar el simulador: aísla el efecto del beamformer de las diferencias de *baseline* (la mezcla degradada ya difiere entre entorno medido y simulado). Si las mejoras `Delta_tot` coinciden entre ISM y MIRD, el simulador predice bien la **ganancia** del procesador aunque los niveles absolutos difieran.

In [ ]:
# MAE(|ISM - MIRD|) sobre la mejora Delta_tot (no sobre el valor absoluto proc)
sub_d = cmp[(cmp.kind == 'Delta_tot') & (cmp.ref == 'early')]
print('MAE(|ISM - MIRD|) por métrica sobre la MEJORA Delta_tot (ref=early):')
print(sub_d.groupby('metric')['ISM-MIRD'].apply(lambda x: np.mean(np.abs(x))).round(3).to_string())

# Comparativo lado a lado: MAE sobre proc (absoluto) vs MAE sobre Delta_tot (mejora)
mae_proc  = (cmp[(cmp.kind == 'proc')      & (cmp.ref == 'early')]
             .groupby('metric')['ISM-MIRD'].apply(lambda x: np.mean(np.abs(x))))
mae_delta = (cmp[(cmp.kind == 'Delta_tot') & (cmp.ref == 'early')]
             .groupby('metric')['ISM-MIRD'].apply(lambda x: np.mean(np.abs(x))))
mae_tbl = pd.DataFrame({'MAE proc (absoluto)': mae_proc, 'MAE Delta_tot (mejora)': mae_delta}).reindex(METRICS)
print('\nComparación MAE absoluto vs MAE mejora (ref=early):')
print(mae_tbl.round(3).to_string())

In [ ]:
# Diferencia ISM - MIRD de la MEJORA del procesador, por procesador/métrica y referencia
fig, axes = plt.subplots(1, len(EVAL_REFS), figsize=(4.2 * len(EVAL_REFS), 3.8), squeeze=False)
for ax, ref in zip(axes[0], EVAL_REFS):
    d = cmp[(cmp['kind'] == 'Delta_tot') & (cmp['ref'] == ref)]
    piv = d.pivot(index='metric', columns='processor', values='ISM-MIRD').reindex(METRICS)
    piv.plot(kind='bar', ax=ax, legend=(ref == EVAL_REFS[0]))
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'ISM - MIRD en MEJORA (ref={ref})')
    ax.set_ylabel('Δ (mejora ISM - mejora MIRD)')
    ax.grid(axis='y', ls=':', alpha=0.6)
fig.suptitle('Discrepancia de la MEJORA del procesador: simulado - medido (0 = réplica perfecta)', y=1.02)
fig.tight_layout()
plt.show()

## 7. Lectura / próximos pasos

- Barras **MIRD vs ISM** cercanas ⇒ el simulador reproduce el comportamiento del entorno medido en ese punto.
- Diferencias sistemáticas en **SIR/SDR** suelen venir de la directividad de la fuente (MIRD usa altavoz directivo; pyroomacoustics es omnidireccional) y de la cola difusa; en **STOI/PESQ** el desvío suele ser menor.
- **Escalado:** una vez validado este punto, ampliar `param_grid`/config a varios `rt60` (0.160/0.360/0.610), `target_angle` (0..90° cada 15°) y spacings, y agregar los gráficos agregados (p. ej. discrepancia media por RT / por ángulo).